# Notebook 00 — Data Loading and Atlas Assembly

**Project:** RetinoblastomaAtlas  
**Author:** Md. Jubayer Hossain  
**Input:** `data/raw/GSE168434_RAW.tar`, `data/raw/GSE249995_RAW.tar`  
**Output:** `data/processed/01_merged_raw.h5ad`, `results/tables/sample_metadata.csv`

---

## Scientific Background

Retinoblastoma (RB) is the most common intraocular malignancy of childhood, arising from cone precursor cells in the developing retina following biallelic *RB1* inactivation (Singh et al. 2018, PNAS). This atlas integrates two publicly available single-cell RNA-seq datasets:

| Dataset | Accession | Cells | Description |
|---------|-----------|-------|-------------|
| Liu et al. 2024 | GSE168434 | ~70,000 | 10 samples from 7 RB patients (RB01–RB07), intraocular only |
| Wan et al. 2025 | GSE249995 | ~30,000 | 4 samples with matched intraocular/extraocular pairs |

By harmonizing these datasets, we can compare the transcriptional landscape of tumour cells at different stages of disease — from the primary intraocular mass to locally invasive extraocular extension along the optic nerve.

## Memory-Efficiency Strategy

This notebook uses four techniques to avoid loading the full ~100k-cell atlas into RAM simultaneously:

1. **Sequential per-sample loading** — each sample is loaded, annotated, and written to a temp `.h5ad` before the next sample is loaded
2. **uint16 dtype** — UMI counts stored as 2-byte integers (vs 4-byte float32), halving memory footprint
3. **Gene universe pre-computation** — feature files are read before matrices to allow per-sample reindexing without a large in-memory outer join
4. **`concat_on_disk()`** — anndata ≥0.10 streams chunks from disk without loading the full atlas into RAM

## Parameters

Adjust these before running if needed.

## Setup

In [1]:
from __future__ import annotations

import gc
import os
import shutil
import tarfile
import tempfile
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

ROOT     = Path("..").resolve()
RAW_DIR  = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
TAB_DIR  = ROOT / "results" / "tables"
OUT_H5AD = PROC_DIR / "01_merged_raw.h5ad"

PROC_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {ROOT}")
print(f"Raw data dir : {RAW_DIR}")
print(f"Output h5ad  : {OUT_H5AD}")

Project root : /mnt/h/Projects/scRNA-Seq/RetinoblastomaAtlas
Raw data dir : /mnt/h/Projects/scRNA-Seq/RetinoblastomaAtlas/data/raw
Output h5ad  : /mnt/h/Projects/scRNA-Seq/RetinoblastomaAtlas/data/processed/01_merged_raw.h5ad


## Sample Manifest

Each sample is annotated with clinical metadata. GSE168434 comprises 10 samples from 7 patients, all intraocular primary tumours. GSE249995 adds 4 samples from 2 patients with matched intraocular and extraocular sites — the latter being the key comparison for studying optic nerve invasion.

In [2]:
GSE168434_SAMPLES = [
    {"gsm": "GSM5139852_RB01_rep1", "patient_id": "RB01", "replicate": "rep1", "disease_stage": "intraocular", "tissue": "primary_tumour"},
    {"gsm": "GSM5139853_RB01_rep2", "patient_id": "RB01", "replicate": "rep2", "disease_stage": "intraocular", "tissue": "primary_tumour"},
    {"gsm": "GSM5139854_RB02_rep1", "patient_id": "RB02", "replicate": "rep1", "disease_stage": "intraocular", "tissue": "primary_tumour"},
    {"gsm": "GSM5139855_RB02_rep2", "patient_id": "RB02", "replicate": "rep2", "disease_stage": "intraocular", "tissue": "primary_tumour"},
    {"gsm": "GSM5139856_RB03_rep1", "patient_id": "RB03", "replicate": "rep1", "disease_stage": "intraocular", "tissue": "primary_tumour"},
    {"gsm": "GSM5139857_RB03_rep2", "patient_id": "RB03", "replicate": "rep2", "disease_stage": "intraocular", "tissue": "primary_tumour"},
    {"gsm": "GSM5139858_RB04",      "patient_id": "RB04", "replicate": "rep1", "disease_stage": "intraocular", "tissue": "primary_tumour"},
    {"gsm": "GSM5139859_RB05",      "patient_id": "RB05", "replicate": "rep1", "disease_stage": "intraocular", "tissue": "primary_tumour"},
    {"gsm": "GSM5139860_RB06",      "patient_id": "RB06", "replicate": "rep1", "disease_stage": "intraocular", "tissue": "primary_tumour"},
    {"gsm": "GSM5139861_RB07",      "patient_id": "RB07", "replicate": "rep1", "disease_stage": "intraocular", "tissue": "primary_tumour"},
]

GSE249995_SAMPLES = [
    {"gsm": "GSM7968797", "sample_id": "S1_in1", "patient_id": "P1", "replicate": "rep1", "disease_stage": "intraocular",  "tissue": "primary_tumour"},
    {"gsm": "GSM7968798", "sample_id": "S2_in2", "patient_id": "P2", "replicate": "rep1", "disease_stage": "intraocular",  "tissue": "primary_tumour"},
    {"gsm": "GSM7968799", "sample_id": "S3_ex1", "patient_id": "P1", "replicate": "rep1", "disease_stage": "extraocular", "tissue": "optic_nerve"},
    {"gsm": "GSM7968800", "sample_id": "S4_ex2", "patient_id": "P2", "replicate": "rep1", "disease_stage": "extraocular", "tissue": "optic_nerve"},
]

all_samples = len(GSE168434_SAMPLES) + len(GSE249995_SAMPLES)
print(f"Total samples: {all_samples} ({len(GSE168434_SAMPLES)} from GSE168434, {len(GSE249995_SAMPLES)} from GSE249995)")

Total samples: 14 (10 from GSE168434, 4 from GSE249995)


## Step 1 — Extract Raw GEO Archives

GEO distributes data as `.tar` archives. GSE168434 uses nested tar.gz (one per sample), while GSE249995 uses a flat structure with GSM-prefixed files. We handle both automatically.

Skip this step by setting `DO_EXTRACT = False` if you have already extracted the archives.

In [3]:
# --- Parameters ---
DO_EXTRACT = True      # Set to False if tarballs are already extracted
GENE_JOIN  = "outer"   # 'outer' = union of all genes; 'inner' = intersection only

In [ ]:
def extract_tar(tar_path: Path, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    mode = "r:gz" if str(tar_path).endswith(".gz") else "r"
    print(f"  Extracting {tar_path.name} → {dest_dir.name}/")
    with tarfile.open(tar_path, mode) as tar:
        try:
            tar.extractall(path=dest_dir, filter="data")
        except TypeError:
            tar.extractall(path=dest_dir)  # Python < 3.12


def _flatten_sample_dir(sample_dir: Path) -> None:
    """Move barcodes/features/matrix files from deep Velocyto paths to sample root."""
    target_files = {"barcodes.tsv.gz", "features.tsv.gz", "matrix.mtx.gz",
                    "barcodes.tsv",    "features.tsv",    "matrix.mtx"}
    for root, dirs, files in os.walk(sample_dir):
        root_path = Path(root)
        if root_path == sample_dir:
            continue
        for f in files:
            if f in target_files:
                src = root_path / f
                dst = sample_dir / f
                if not dst.exists():
                    shutil.move(str(src), str(dst))
    home_dir = sample_dir / "home"
    if home_dir.exists():
        shutil.rmtree(home_dir, ignore_errors=True)


def extract_gse168434(raw_dir: Path) -> None:
    outer_tar = raw_dir / "GSE168434_RAW.tar"
    gse_dir   = raw_dir / "GSE168434"
    if not outer_tar.exists():
        raise FileNotFoundError(f"{outer_tar} not found. Download from GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE168434")
    extract_tar(outer_tar, gse_dir)
    for tar_gz in sorted(gse_dir.glob("*.tar.gz")):
        sample_name = tar_gz.name.split("_counts")[0]
        sample_dir  = gse_dir / sample_name
        if sample_dir.exists() and (sample_dir / "matrix.mtx.gz").exists():
            print(f"    {sample_name}: already extracted")
            continue
        sample_dir.mkdir(exist_ok=True)
        extract_tar(tar_gz, sample_dir)
        _flatten_sample_dir(sample_dir)
    print(f"  GSE168434 extraction complete")


def extract_gse249995(raw_dir: Path) -> None:
    outer_tar = raw_dir / "GSE249995_RAW.tar"
    gse_dir   = raw_dir / "GSE249995"
    if not outer_tar.exists():
        raise FileNotFoundError(f"{outer_tar} not found. Download from GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE249995")
    if gse_dir.exists() and any(gse_dir.glob("GSM*matrix*")):
        print("  GSE249995: already extracted")
        return
    extract_tar(outer_tar, gse_dir)
    print(f"  GSE249995 extraction complete")


if DO_EXTRACT:
    print("Step 1 — Extracting raw GEO archives")
    extract_gse168434(RAW_DIR)
    extract_gse249995(RAW_DIR)
else:
    print("Skipping extraction (DO_EXTRACT=False)")

## Step 2 — Compute Gene Universe

Before loading any count matrix, we read all feature files (small, <1 MB each) to determine the union of all genes across samples. This allows each sample to be reindexed to the full gene space *during loading*, avoiding the large in-memory expansion that `anndata.concat(join='outer')` would require.

We use `join='outer'` (gene union) to preserve all genes including mitochondrial genes that may be absent from one dataset due to differences in Cell Ranger annotation versions.

In [4]:
def read_feature_names(feature_file: Path) -> pd.Series:
    return pd.read_csv(feature_file, sep="\t", header=None, usecols=[1])[1]


def compute_gene_universe(sample_feature_files: list, join: str = "outer") -> list:
    print(f"  Computing gene universe (join='{join}') from {len(sample_feature_files)} feature files...")
    gene_sets = []
    for f in sample_feature_files:
        genes = read_feature_names(f)
        gene_sets.append(set(genes.values))
        print(f"    {f.parent.name or f.name}: {len(genes):,} genes")
    if join == "inner":
        universe = sorted(set.intersection(*gene_sets))
    else:
        universe = sorted(set.union(*gene_sets))
    print(f"  Gene universe ({join}): {len(universe):,} genes")
    return universe


print("Step 2 — Collecting feature files and computing gene universe")
feature_files = []
gse168434_dir = RAW_DIR / "GSE168434"
for meta in GSE168434_SAMPLES:
    for ext in ["features.tsv.gz", "features.tsv"]:
        f = gse168434_dir / meta["gsm"] / ext
        if f.exists():
            feature_files.append(f)
            break

gse249995_dir = RAW_DIR / "GSE249995"
for meta in GSE249995_SAMPLES:
    matches = list(gse249995_dir.glob(f"{meta['gsm']}*features*"))
    if matches:
        feature_files.append(matches[0])

print(f"  Found {len(feature_files)} feature files")
gene_universe = compute_gene_universe(feature_files, join=GENE_JOIN)

Step 2 — Collecting feature files and computing gene universe
  Found 14 feature files
  Computing gene universe (join='outer') from 14 feature files...
    GSM5139852_RB01_rep1: 33,538 genes
    GSM5139853_RB01_rep2: 33,538 genes
    GSM5139854_RB02_rep1: 33,538 genes
    GSM5139855_RB02_rep2: 33,538 genes
    GSM5139856_RB03_rep1: 33,538 genes
    GSM5139857_RB03_rep2: 33,538 genes
    GSM5139858_RB04: 33,538 genes
    GSM5139859_RB05: 33,538 genes
    GSM5139860_RB06: 33,538 genes
    GSM5139861_RB07: 33,538 genes
    GSE249995: 36,601 genes
    GSE249995: 60,656 genes
    GSE249995: 33,538 genes
    GSE249995: 60,656 genes
  Gene universe (outer): 61,108 genes


## Step 3 — Load Samples Sequentially (Memory-Efficient)

Each sample is loaded one at a time using `sc.read_10x_mtx()`, immediately cast to `uint16` dtype, reindexed to the gene universe, and written to a temporary `.h5ad` file. The in-memory AnnData is then explicitly deleted and garbage-collected before loading the next sample.

**Why uint16?** Raw UMI counts from 10x Genomics rarely exceed 65,535 per gene per cell. Storing as `uint16` (2 bytes) instead of `float32` (4 bytes) halves the non-zero data array size.

In [5]:
def load_10x_sample_uint16(mtx_dir: Path, sample_meta: dict, gene_universe: list) -> ad.AnnData:
    adata = sc.read_10x_mtx(mtx_dir, var_names="gene_symbols", cache=False, make_unique=True)
    # Cast to uint16 immediately
    if sp.issparse(adata.X):
        adata.X = adata.X.astype(np.uint16)
    else:
        adata.X = adata.X.astype(np.uint16)
    # Restrict to gene universe and fill missing genes with zeros
    adata = adata[:, adata.var_names.isin(gene_universe)].copy()
    missing_genes = [g for g in gene_universe if g not in adata.var_names]
    if missing_genes:
        zero_block = sp.csr_matrix((adata.n_obs, len(missing_genes)), dtype=np.uint16)
        missing_adata = ad.AnnData(
            X=zero_block,
            obs=pd.DataFrame(index=adata.obs_names),
            var=pd.DataFrame(index=missing_genes),
        )
        adata = ad.concat([adata, missing_adata], axis=1, join="outer")
    adata = adata[:, gene_universe].copy()
    sample_id = sample_meta.get("sample_id", sample_meta.get("gsm", "unknown"))
    adata.obs_names = [f"{bc}-{sample_id}" for bc in adata.obs_names]
    for col, val in sample_meta.items():
        adata.obs[col] = val
    if "gsm" in adata.obs.columns and "sample_id" not in sample_meta:
        adata.obs.rename(columns={"gsm": "sample_id"}, inplace=True)
    print(f"    {sample_id}: {adata.n_obs:,} cells × {adata.n_vars:,} genes")
    return adata


def load_gse249995_flat(gsm_id: str, gse_dir: Path, sample_meta: dict, gene_universe: list) -> ad.AnnData:
    def _find(pattern: str) -> Path:
        matches = list(gse_dir.glob(f"{gsm_id}*{pattern}*"))
        if not matches:
            raise FileNotFoundError(f"No file matching '{gsm_id}*{pattern}*' in {gse_dir}")
        return matches[0]
    mtx_src      = _find("matrix.mtx")
    barcodes_src = _find("barcodes.tsv")
    features_src = _find("features.tsv")
    tmp = Path(tempfile.mkdtemp(prefix=f"gsm_{gsm_id}_"))
    try:
        suffix_m = ".gz" if str(mtx_src).endswith(".gz") else ""
        suffix_b = ".gz" if str(barcodes_src).endswith(".gz") else ""
        suffix_f = ".gz" if str(features_src).endswith(".gz") else ""
        shutil.copy2(mtx_src,      tmp / f"matrix.mtx{suffix_m}")
        shutil.copy2(barcodes_src, tmp / f"barcodes.tsv{suffix_b}")
        shutil.copy2(features_src, tmp / f"features.tsv{suffix_f}")
        return load_10x_sample_uint16(tmp, sample_meta, gene_universe)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


print("Step 3 — Loading samples sequentially and writing temp h5ad files")
tmp_dir   = PROC_DIR / "_tmp_samples"
tmp_dir.mkdir(exist_ok=True)
tmp_h5ads = []
sample_idx = 0

print(f"\n  --- GSE168434 ({len(GSE168434_SAMPLES)} samples) ---")
for meta in GSE168434_SAMPLES:
    sample_dir = gse168434_dir / meta["gsm"]
    if not sample_dir.exists():
        print(f"  WARNING: {sample_dir} not found — skipping")
        continue
    sample_meta = dict(meta)
    if "sample_id" not in sample_meta:
        sample_meta["sample_id"] = meta["gsm"]
    sample_meta["dataset"] = "GSE168434"
    try:
        adata_s  = load_10x_sample_uint16(sample_dir, sample_meta, gene_universe)
        out_path = tmp_dir / f"sample_{sample_idx:03d}.h5ad"
        adata_s.write_h5ad(out_path, compression="gzip", compression_opts=4)
        tmp_h5ads.append(out_path)
        sample_idx += 1
    except Exception as e:
        print(f"  ERROR loading {meta['gsm']}: {e}")
        continue
    finally:
        try:
            del adata_s
        except NameError:
            pass
        gc.collect()

print(f"\n  --- GSE249995 ({len(GSE249995_SAMPLES)} samples) ---")
for meta in GSE249995_SAMPLES:
    sample_meta = dict(meta)
    sample_meta["dataset"] = "GSE249995"
    try:
        adata_s  = load_gse249995_flat(meta["gsm"], gse249995_dir, sample_meta, gene_universe)
        out_path = tmp_dir / f"sample_{sample_idx:03d}.h5ad"
        adata_s.write_h5ad(out_path, compression="gzip", compression_opts=4)
        tmp_h5ads.append(out_path)
        sample_idx += 1
    except Exception as e:
        print(f"  ERROR loading {meta['gsm']}: {e}")
        continue
    finally:
        try:
            del adata_s
        except NameError:
            pass
        gc.collect()

print(f"\n  {len(tmp_h5ads)} samples written to temp h5ad files")

Step 3 — Loading samples sequentially and writing temp h5ad files

  --- GSE168434 (10 samples) ---
    GSM5139852_RB01_rep1: 6,990 cells × 61,108 genes
    GSM5139853_RB01_rep2: 6,826 cells × 61,108 genes
    GSM5139854_RB02_rep1: 4,185 cells × 61,108 genes
    GSM5139855_RB02_rep2: 2,140 cells × 61,108 genes
    GSM5139856_RB03_rep1: 7,596 cells × 61,108 genes
    GSM5139857_RB03_rep2: 7,638 cells × 61,108 genes
    GSM5139858_RB04: 14,093 cells × 61,108 genes
    GSM5139859_RB05: 13,016 cells × 61,108 genes
    GSM5139860_RB06: 14,407 cells × 61,108 genes
    GSM5139861_RB07: 14,681 cells × 61,108 genes

  --- GSE249995 (4 samples) ---
    S1_in1: 10,453 cells × 61,108 genes
    S2_in2: 15,386 cells × 61,108 genes
    S3_ex1: 11,099 cells × 61,108 genes
    S4_ex2: 13,370 cells × 61,108 genes

  14 samples written to temp h5ad files


## Step 4 — Disk-Based Concatenation

`anndata.experimental.concat_on_disk()` (anndata ≥ 0.10) concatenates `.h5ad` files by streaming chunks from disk directly into the output file — without ever loading the full atlas into RAM. Peak memory = one chunk, not the entire atlas.

A fallback to in-memory `anndata.concat()` is provided for older anndata versions.

In [6]:
print("Step 4 — Concatenating on disk")

try:
    from anndata.experimental import concat_on_disk
    print("  Using anndata.experimental.concat_on_disk() — peak RAM bounded by single chunk")
    concat_on_disk(
        in_files={p.stem: p for p in tmp_h5ads},
        out_file=OUT_H5AD,
        axis=0,
        join="outer",
        label="sample_key",
        index_unique=None,
        fill_value=0,
    )
    adata = ad.read_h5ad(OUT_H5AD)
except ImportError:
    print("  Falling back to in-memory concat (anndata < 0.10)")
    adatas = [ad.read_h5ad(p) for p in tmp_h5ads]
    adata  = ad.concat(adatas, axis=0, join="outer", fill_value=0, index_unique=None)
    del adatas; gc.collect()
    adata.write_h5ad(OUT_H5AD, compression="gzip", compression_opts=4)

print(f"  Atlas: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

Step 4 — Concatenating on disk
  Using anndata.experimental.concat_on_disk() — peak RAM bounded by single chunk
  Atlas: 141,880 cells × 61,108 genes


## Step 5 — Add Gene QC Flags

Three Boolean flags are added to `adata.var` (gene metadata) to enable QC metric computation in the next notebook:

- **`mt`**: Mitochondrial genes (`MT-` prefix) — elevated in dying/damaged cells
- **`ribo`**: Ribosomal protein genes (`RPS`, `RPL` prefixes) — high in stressed or rapidly proliferating cells
- **`hb`**: Hemoglobin genes (`HB*`) — indicator of red blood cell contamination from ocular vasculature

In [7]:
print("Step 5 — Adding gene QC flags")
adata.var["mt"]   = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"]   = adata.var_names.str.match(r"^HB[^(P)]")

# Remove helper column from concat
if "sample_key" in adata.obs.columns:
    adata.obs.drop(columns=["sample_key"], inplace=True)

print(f"  Mitochondrial genes (MT-):  {adata.var['mt'].sum():,}")
print(f"  Ribosomal genes (RPS/RPL):  {adata.var['ribo'].sum():,}")
print(f"  Hemoglobin genes (HB*):     {adata.var['hb'].sum():,}")

Step 5 — Adding gene QC flags
  Mitochondrial genes (MT-):  37
  Ribosomal genes (RPS/RPL):  1,349
  Hemoglobin genes (HB*):     14


## Step 6 — Sample Metadata Summary

In [8]:
print("Step 6 — Sample metadata summary")
group_cols = [c for c in ["sample_id", "dataset", "patient_id", "disease_stage", "tissue", "replicate"]
              if c in adata.obs.columns]

meta = (
    adata.obs[group_cols]
    .drop_duplicates()
    .merge(
        adata.obs.groupby("sample_id").size().rename("n_cells").reset_index(),
        on="sample_id",
    )
    .sort_values("sample_id")
    .reset_index(drop=True)
)
meta.to_csv(TAB_DIR / "sample_metadata.csv", index=False)
print(meta.to_string(index=False))

Step 6 — Sample metadata summary
           sample_id   dataset patient_id disease_stage         tissue replicate  n_cells
GSM5139852_RB01_rep1 GSE168434       RB01   intraocular primary_tumour      rep1     6990
GSM5139853_RB01_rep2 GSE168434       RB01   intraocular primary_tumour      rep2     6826
GSM5139854_RB02_rep1 GSE168434       RB02   intraocular primary_tumour      rep1     4185
GSM5139855_RB02_rep2 GSE168434       RB02   intraocular primary_tumour      rep2     2140
GSM5139856_RB03_rep1 GSE168434       RB03   intraocular primary_tumour      rep1     7596
GSM5139857_RB03_rep2 GSE168434       RB03   intraocular primary_tumour      rep2     7638
     GSM5139858_RB04 GSE168434       RB04   intraocular primary_tumour      rep1    14093
     GSM5139859_RB05 GSE168434       RB05   intraocular primary_tumour      rep1    13016
     GSM5139860_RB06 GSE168434       RB06   intraocular primary_tumour      rep1    14407
     GSM5139861_RB07 GSE168434       RB07   intraocular primary_tum

## Step 7 — Save Final Atlas

In [10]:
print("Step 7 — Writing final atlas")
adata.write_h5ad(OUT_H5AD, compression="gzip", compression_opts=4)
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  Saved: {OUT_H5AD.name}  ({adata.n_obs:,} cells × {adata.n_vars:,} genes, {size_mb:.0f} MB)")

# Cleanup temp files
shutil.rmtree(tmp_dir, ignore_errors=True)
print("  Temp files cleaned up.")

print("\n  AnnData structure:")
print(f"    .X                   : raw UMI counts (CSR sparse, uint16→int32)")
print(f"    .obs['sample_id']    : sample identifier")
print(f"    .obs['dataset']      : GSE168434 | GSE249995")
print(f"    .obs['patient_id']   : RB01-RB07 | P1-P2")
print(f"    .obs['disease_stage']: intraocular | extraocular")
print(f"    .var['mt']           : mitochondrial gene flag")
print(f"    .var['ribo']         : ribosomal gene flag")
print(f"    .var['hb']           : hemoglobin gene flag")
print(f"\n  Next: Run notebook 01_qc_filtering.ipynb")

Step 7 — Writing final atlas
  Saved: 01_merged_raw.h5ad  (141,880 cells × 61,108 genes, 962 MB)
  Temp files cleaned up.

  AnnData structure:
    .X                   : raw UMI counts (CSR sparse, uint16→int32)
    .obs['sample_id']    : sample identifier
    .obs['dataset']      : GSE168434 | GSE249995
    .obs['patient_id']   : RB01-RB07 | P1-P2
    .obs['disease_stage']: intraocular | extraocular
    .var['mt']           : mitochondrial gene flag
    .var['ribo']         : ribosomal gene flag
    .var['hb']           : hemoglobin gene flag

  Next: Run notebook 01_qc_filtering.ipynb
